# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rimlazrek1/flyrank-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Setup (Local)

In [9]:
import os
from pathlib import Path

import duckdb

ROOT = Path.cwd()
while not (ROOT / "data" / "raw").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    import getpass
    print("Tip: pip install python-dotenv")

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass("HF READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

print("Connected.")

Connected.


## 1. My rule and its reason codes

**Lane 4 — CTR / Engagement Opportunity Scoring**

Same slice as ML-04: warehouse **March 2026** (`month=2026-03`), one row per page, `imp_mar >= 100`, real position only.

### Two signal checks

Before scoring, we check the two ideas our rule depends on (same logic FlyRank uses for CTR flags — we test the **numbers**, not a product flag column):

| Signal | What we check | FlyRank idea it mirrors |
|---|---|---|
| **1. CTR vs position** | CTR changes by `position_tier` | `low_ctr_visible_page` — rank band matters |
| **2. Volume** | Low-impression pages are noisier | impression floor on visible pages |

### My rule

> Rank pages **higher** when they have **enough March impressions** and their **`ctr_mar` is below the median for their `position_tier`**. A reviewer opens the top of that list first.

This reuses the rule from ML-03 / ML-04. We only claim **observed / directional / decision-support**, not that a fix caused more clicks.

### Reason codes (one per row)

| Reason code | When (first match wins) |
|---|---|
| `high_visibility_ctr_gap` | below tier median **and** `imp_mar >= 500` |
| `ctr_below_tier_median` | below tier median |
| `general_ctr_monitor` | everything else on the list |

### Signal verdicts

**Signal 1 — CTR vs position:** `CONFIRMED`  
Weighted CTR falls from 0.41% on `top_3` to 0.04% on `deep` (n = 8,295 and 3,949 pages), so CTR must be compared within `position_tier`, not site-wide.

**Signal 2 — Volume:** `CONFIRMED`  
Low-impression bands are noisier: 74% zero-click pages in `100-299` (n = 26,775) vs 3% in `3000+` (n = 22,157), so a minimum impression floor is needed before we trust CTR.

In [10]:
import pandas as pd

TIER_ORDER = ["top_3", "page_1", "striking", "page_3_5", "deep"]
IMP_BANDS = [100, 300, 500, 1000, 3000, float("inf")]
IMP_LABELS = ["100-299", "300-499", "500-999", "1000-2999", "3000+"]

features = con.sql(f"""
    WITH daily AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS imp_mar,
            SUM(gsc_clicks) AS clk_mar,
            AVG(NULLIF(gsc_avg_position, 0)) AS pos_avg_mar
        FROM {FACT_MAR}
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 100
    ),
    scored AS (
        SELECT
            *,
            CASE WHEN imp_mar > 0 THEN 100.0 * clk_mar / imp_mar END AS ctr_mar,
            CASE
                WHEN pos_avg_mar <= 3 THEN 'top_3'
                WHEN pos_avg_mar <= 10 THEN 'page_1'
                WHEN pos_avg_mar <= 20 THEN 'striking'
                WHEN pos_avg_mar <= 50 THEN 'page_3_5'
                ELSE 'deep'
            END AS position_tier
        FROM daily
        WHERE pos_avg_mar > 0
    ),
    labeled AS (
        SELECT
            s.*,
            MEDIAN(ctr_mar) OVER (PARTITION BY position_tier) AS tier_median_ctr,
            CASE
                WHEN ctr_mar < MEDIAN(ctr_mar) OVER (PARTITION BY position_tier) THEN 1
                ELSE 0
            END AS is_ctr_underperformer
        FROM scored s
    )
    SELECT * FROM labeled
""").df()

print(f"Lane slice n = {len(features):,}")

print("\nSIGNAL 1 — CTR vs position (by position_tier)")
sig1 = (
    features.groupby("position_tier", observed=True)
    .agg(
        n=("content_hash_id", "count"),
        sum_imp=("imp_mar", "sum"),
        sum_clk=("clk_mar", "sum"),
    )
    .assign(weighted_ctr_mar=lambda d: 100.0 * d["sum_clk"] / d["sum_imp"])
    .drop(columns=["sum_imp", "sum_clk"])
    .reindex(TIER_ORDER)
    .reset_index()
)
print(sig1.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\nSIGNAL 2 — Volume (by impression band)")
features["imp_band"] = pd.cut(
    features["imp_mar"], bins=IMP_BANDS, labels=IMP_LABELS, right=False
)
sig2 = (
    features.groupby("imp_band", observed=True)
    .agg(
        n=("content_hash_id", "count"),
        share_zero_clicks=("ctr_mar", lambda s: (s == 0).mean()),
    )
    .reset_index()
)
print(sig2.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

Lane slice n = 101,441

SIGNAL 1 — CTR vs position (by position_tier)
position_tier     n  weighted_ctr_mar
        top_3  8295            0.4067
       page_1 46531            0.3232
     striking 21738            0.3052
     page_3_5 20928            0.1361
         deep  3949            0.0356

SIGNAL 2 — Volume (by impression band)
 imp_band     n  share_zero_clicks
  100-299 26775             0.7393
  300-499 12742             0.5642
  500-999 16866             0.3801
1000-2999 22901             0.1624
    3000+ 22157             0.0301


## 2. Build the ranked queue (writes the CSV)

*Write work/outputs/baseline_action_score.csv.*

In [11]:
import numpy as np

OUT = ROOT / "work" / "outputs" / "baseline_action_score.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)

queue = features.copy()
queue["ctr_gap"] = queue["tier_median_ctr"] - queue["ctr_mar"]
queue["baseline_score"] = queue["ctr_gap"] * np.log1p(queue["imp_mar"])

def reason_code(row) -> str:
    if row["is_ctr_underperformer"] == 1 and row["imp_mar"] >= 500:
        return "high_visibility_ctr_gap"
    if row["is_ctr_underperformer"] == 1:
        return "ctr_below_tier_median"
    return "general_ctr_monitor"

def action_label(code: str) -> str:
    if code == "high_visibility_ctr_gap":
        return "refresh_and_review_ctr"
    if code == "ctr_below_tier_median":
        return "review_ctr"
    return "monitor"

queue["reason_code"] = queue.apply(reason_code, axis=1)
queue["action"] = queue["reason_code"].map(action_label)
queue["baseline_rank"] = queue["baseline_score"].rank(method="first", ascending=False).astype(int)

export_cols = [
    "baseline_rank", "content_hash_id", "client_hash_id",
    "imp_mar", "ctr_mar", "pos_avg_mar", "position_tier",
    "baseline_score", "reason_code", "action",
]
queue.sort_values("baseline_rank").to_csv(OUT, index=False, columns=export_cols)
print(f"Wrote {len(queue):,} rows → {OUT}")


Wrote 101,441 rows → c:\Users\rimla\Desktop\work_folder\flyrank-internship\work\outputs\baseline_action_score.csv


### Baseline metrics (For ML-08 model comparison)

**Good pick** = `is_ctr_underperformer` (CTR below tier median — same proxy as ML-03).

**Precision@K** = of the top K pages our rule ranks, how many are good picks?

Also print **base rate** (random picking baseline). Score uses `ctr_gap`, so P@K will look strong — that's expected; the model must beat this without label leakage.

In [14]:
import json

LABEL = "is_ctr_underperformer"

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

labels = queue[LABEL].to_numpy()
scores = queue["baseline_score"].to_numpy()
base_rate = float(labels.mean())

metrics = {
    "method": "baseline_rule",
    "label": LABEL,
    "slice": "month=2026-03",
    "n": int(len(queue)),
    "base_rate": base_rate,
    "precision_at_10": precision_at_k(scores, labels, 10),
    "precision_at_20": precision_at_k(scores, labels, 20),
    "precision_at_50": precision_at_k(scores, labels, 50),
}

METRICS_OUT = ROOT / "work" / "outputs" / "baseline_metrics.json"
METRICS_OUT.write_text(json.dumps(metrics, indent=2))

print(f"Base rate (share {LABEL}=1): {base_rate:.3f}")
for k in (10, 20, 50):
    print(f"Precision@{k}: {metrics[f'precision_at_{k}']:.3f}")
print(f"Saved → {METRICS_OUT}")

Base rate (share is_ctr_underperformer=1): 0.377
Precision@10: 1.000
Precision@20: 1.000
Precision@50: 1.000
Saved → c:\Users\rimla\Desktop\work_folder\flyrank-internship\work\outputs\baseline_metrics.json


## 3. Top-10 review

For each of the top 10: **action**, **why it's there**, and **what would make it wrong**.

| Rank | action | reason_code | why it's here | what would make it wrong |
|---:|---|---|---|---|
| 1 | refresh_and_review_ctr | high_visibility_ctr_gap | `top_3` page with ~38k March impressions but ~0.003% CTR — large gap vs tier peers, high score from volume × ctr_gap | CTR is low because of a featured snippet / brand query where clicks are not expected |
| 2 | refresh_and_review_ctr | high_visibility_ctr_gap | ~24k impressions on `top_3`, CTR ~0.004% — visible page, weak click capture vs same-tier median | Low CTR is intentional (login wall, app store, non-click intent) |
| 3 | refresh_and_review_ctr | high_visibility_ctr_gap | ~60k impressions, `top_3`, CTR ~0.03% — still below tier median with heavy weight on impressions | Clicks are fine for rank; gap is driven by one noisy week in March |
| 4 | refresh_and_review_ctr | high_visibility_ctr_gap | ~26k impressions, `top_3`, CTR ~0.012% — below-tier CTR with enough volume to trust the signal | Page recently changed URL or tracking broke clicks only |
| 5 | refresh_and_review_ctr | high_visibility_ctr_gap | ~13k impressions, `top_3`, **0% CTR** — zero clicks with strong visibility pushes it high | Zero clicks = measurement gap or SERP answers on-page; not a title/meta fix |
| 6 | refresh_and_review_ctr | high_visibility_ctr_gap | ~23k impressions, `top_3`, CTR ~0.017% — underperforms peers at similar rank | Position is inflated by low-competition queries, not real page-1 demand |
| 7 | refresh_and_review_ctr | high_visibility_ctr_gap | ~81k impressions, `top_3`, CTR ~0.043% — score driven by very high impressions more than extreme CTR | CTR is actually near tier norms; reviewer time better spent lower on list |
| 8 | refresh_and_review_ctr | high_visibility_ctr_gap | ~10k impressions, `top_3`, **0% CTR** — zero-click visible `top_3` slot | Same as rank 5: zero-click may be SERP/tracking, not snippet quality |
| 9 | refresh_and_review_ctr | high_visibility_ctr_gap | ~135k impressions, `page_1`, CTR ~0.0007% — massive visibility, almost no clicks vs `page_1` median | Demand is real but intent mismatch (wrong page ranking for query set) |
| 10 | refresh_and_review_ctr | high_visibility_ctr_gap | ~124k impressions, `page_1`, CTR ~0.0008% — same pattern as rank 9 | Low CTR explained by seasonality or site-wide March dip, not this page |

In [12]:
top10_cols = [
    "baseline_rank", "content_hash_id", "imp_mar", "ctr_mar",
    "position_tier", "reason_code", "action",
]
print(queue.sort_values("baseline_rank").head(10)[top10_cols].to_string(index=False))

 baseline_rank          content_hash_id  imp_mar  ctr_mar position_tier             reason_code                 action
             1 content_d61fc394d10cba41  38000.0 0.002632         top_3 high_visibility_ctr_gap refresh_and_review_ctr
             2 content_66bf45eb0c5bb550  24259.0 0.004122         top_3 high_visibility_ctr_gap refresh_and_review_ctr
             3 content_fc67675904376267  60172.0 0.029914         top_3 high_visibility_ctr_gap refresh_and_review_ctr
             4 content_b9acd1ebff7d34ff  25941.0 0.011565         top_3 high_visibility_ctr_gap refresh_and_review_ctr
             5 content_fa17add7836d36c3  12588.0 0.000000         top_3 high_visibility_ctr_gap refresh_and_review_ctr
             6 content_1d7764b642f7bb9f  23402.0 0.017093         top_3 high_visibility_ctr_gap refresh_and_review_ctr
             7 content_306bc78dff1eb683  80821.0 0.043306         top_3 high_visibility_ctr_gap refresh_and_review_ctr
             8 content_d397987113cb84a0   9887.0

## 4. Weak picks + leakage check

**Weak picks** *(1–2 ranks from the top 10 that look shaky and why)*:

1. **Rank 5** — 0% CTR with ~13k impressions may be a SERP/tracking artifact (zero clicks ≠ bad meta). I'd confirm clicks exist in daily data before editing.
2. **Rank 7** — CTR ~0.043% on `top_3` is not extreme; it ranks high mostly because ~81k impressions inflate `baseline_score`. A human might prioritize zero-click rows first.

**Leakage check (plain words):**

- No product flags (`needs_ctr_fix`, `health_score`, …) — not in the data ✓
- No future-window columns — March 2026 only ✓
- No `trend_direction` / `trend_pct` in the score ✓
- `ctr_gap` is **part of the baseline rule**, not a model feature — OK for this hand rule; dropped for ML-08 models (see ML-04 trap)

In [13]:
FORBIDDEN = {"trend_direction", "trend_pct", "health_score", "needs_ctr_fix", "priority_score"}
print("Forbidden cols present:", sorted(FORBIDDEN & set(queue.columns)) or "none (good)")

Forbidden cols present: none (good)


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.